In [ ]:
import env
from CADGNTrainer import CADGNTrainer, Stage1Config, Stage2Config

from losses import _generate_candidate_pairs
from models_config import models
from ds.cladder import CLadderSample, CLadderDataset, load_cladder_v1_5, CLadderLoaderConfig
from cadgn import CADGNCore, TokenizerFamily, flush_gpu, save_history
from search import ArchParams

# Unfortunately, the current implementation precludes saving modified versions of the various modules (config is static)

flush_gpu()

# rounded to 4dp where appropriate
best_params = {
    "ca_dgn_dim": 256,
    "max_layers": 5,
    "num_iters": 4,
    "epsilon": 0.0665, # 0.06646303815100524
    "base_gamma": 0.0296, # 0.02956587470709161
    "encoder_dropout": 0.0563,  # 0.05627028512612424
    "decoder_expansion": 4,
    "decoder_dropout": 0.1743,  # 0.17434218753510897
    "head_encoder_dropout": 0.0595,  # 0.05948165802005359
    "head_decoder_dropout": 0.0543,  # 0.0543472754267514
}

arch = ArchParams.from_dict(best_params)

s1c = Stage1Config(
    lr=0.0015,  # 0.0015121612751471838
    weight_decay=0.0004,  # 0.0004379467905083944
    grad_accum_steps=13,
    # Default
    max_steps_per_epoch=550,
    max_steps_per_epoch_val=150,
    w_mmd=1.0
)

s2c = Stage2Config(
    lr=0.0015,  # 0.0015121612751471838
    weight_decay=0.0004,  # 0.0004379467905083944
    grad_accum_steps=13,
    max_steps_per_epoch=550,
    max_steps_per_epoch_val=150,
    w_mmd = 1.0,
)

"""
 Models to Ablate for:
    - LLM alone (prompt + gate cls)
    - GCN  (does the graph help?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False
    - ADGN (Gravina et al., 2023) (does Antisymmetry help?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False
    - CADGN (does a directed neighbourhood aggregation help?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False

    - CADGN (remove CADGN Decoder, is it necessary?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False

    - GATv2Conv (Brody et al., 2022) (does Antisymmetry + RoPE directionality help over learned attention?)
        - Stage 1 (retrain): w_mmd=1.0 / w_mmd=0.0
        - Stage 2: graph_first=True/False
    -

"""




In [ ]:
from ablations.BaselineLLMTrainer import BaselineLLMTrainer, BaselineSampleResult
# todo: check whether we need to integrate these baseline sample info into the standard trainer pipeline.
from ablations.ADGNConv import ADGNConvWrapper
from ablations.gat_encoder import GATEncoder
